# Tutorial 01: Basic Factor Evaluation

This tutorial covers the fundamentals of creating and evaluating factors using the factor_engine.

## What You'll Learn
- Create simple factor expressions using the DSL API
- Run factors with DebugBackend for testing
- Understand factor structure and output format
- Basic troubleshooting

In [ ]:
import sys
import numpy as np
import pandas as pd

# Add factor_engine to path
sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import display_success, display_warning, display_factor_result, NotebookTimer

## Step 1: Import Factor Engine Components

The main components you need:
- `api`: DSL functions like `col()`, `rank()`, `ts_mean()`
- `Factor`: Container for factor expressions
- `DebugBackend`: Testing backend that generates synthetic data

In [ ]:
from api import col, rank, ts_mean, ts_std, delay, zscore, Factor
from backend.debug_backend import DebugBackend
from runtime.engine import FactorEngine
from storage.datasource import DataSource

display_success("Successfully imported factor_engine components")

## Step 2: Create a Simple Factor Expression

Let's build a momentum factor: 20-day moving average minus 5-day moving average, then ranked.

In [ ]:
# Build the expression
expr = rank(ts_mean(col("close"), 20) - ts_mean(col("close"), 5))

# Wrap it in a Factor object
momentum_factor = Factor(
    name="momentum_20_5_rank",
    expr=expr,
    freq="1d",
    universe="equities"
)

print(f"Factor created: {momentum_factor.name}")
print(f"Expression type: {type(expr)}")

## Step 3: Set Up the Engine

The `DebugBackend` generates random panel data for testing without needing real data.

In [ ]:
class DemoDataSource(DataSource):
    """Minimal data source - DebugBackend doesn't use it."""
    def load_column(self, name: str):
        raise NotImplementedError("Use DebugBackend for demos")

# Create engine with debug backend
engine = FactorEngine(
    backend=DebugBackend(),
    data_source=DemoDataSource()
)

display_success("Engine initialized with DebugBackend")

## Step 4: Run the Factor

Execute the factor and examine the output.

In [ ]:
with NotebookTimer("Factor evaluation"):
    result = engine.run(momentum_factor)

# Extract the result DataFrame
factor_values = result["result"]

print(f"Result shape: {factor_values.shape}")
print(f"Index type: {type(factor_values.index)}")
print(f"\nFirst few rows:")
display_factor_result(factor_values.head(20), "Momentum Factor Output")

## Step 5: Analyze the Output

Factor outputs are typically MultiIndex Series (date, ticker) → value.

In [ ]:
# Basic statistics
print("Factor Statistics:")
print(f"  Mean: {factor_values.mean():.4f}")
print(f"  Std:  {factor_values.std():.4f}")
print(f"  Min:  {factor_values.min():.4f}")
print(f"  Max:  {factor_values.max():.4f}")
print(f"  Null count: {factor_values.isna().sum()}")

# Cross-sectional distribution on one date
if isinstance(factor_values.index, pd.MultiIndex):
    first_date = factor_values.index.get_level_values(0)[0]
    cross_section = factor_values.xs(first_date, level=0)
    print(f"\nCross-section on {first_date}:")
    print(f"  Stocks: {len(cross_section)}")
    print(f"  Range: [{cross_section.min():.2f}, {cross_section.max():.2f}]")

## Step 6: Try Different Factor Expressions

Experiment with other operators.

In [ ]:
# Volatility factor
volatility_expr = ts_std(col("close"), 20)
volatility_factor = Factor("volatility_20d", volatility_expr, "1d", "equities")

vol_result = engine.run(volatility_factor)
print("Volatility factor completed")
print(f"Mean volatility: {vol_result['result'].mean():.4f}")

# Z-score of returns
zscore_expr = zscore(col("close") / delay(col("close"), 1) - 1.0)
zscore_factor = Factor("returns_zscore", zscore_expr, "1d", "equities")

zscore_result = engine.run(zscore_factor)
print("\nZ-score factor completed")
print(f"Mean: {zscore_result['result'].mean():.4f}")
print(f"Std:  {zscore_result['result'].std():.4f}")

## Key Takeaways

1. **DSL API**: Use `col()` to reference fields, operators like `ts_mean()`, `rank()` to build expressions
2. **Factor**: Wraps expressions with metadata (name, frequency, universe)
3. **Engine**: Executes factors; DebugBackend is perfect for prototyping
4. **Output**: MultiIndex Series (date, ticker) with factor values

## Next Steps

- Tutorial 02: Data preprocessing and handling real data
- Tutorial 03: Factor optimization and parameter tuning
- Tutorial 04: Multi-factor selection and combination